In [ ]:
source("helpers/init.r")

In [ ]:
cohorts  <- fread(paste0(READY_DATA, "cohorts.txt"))
patients <- cohorts %>% filter(!ref) %>% pull(sampleId)

# 0 - Load data

In [ ]:
ready <-
  runQuery("circos", sampleIds = patients) %>%
  mutate(end = start + 1000000, chr = sub("hschr", "hs", chr)) %>%
  group_by(chr, start, end) %>%
  summarise(
    across(c(homozygous_del, neutral_loss_and_loh, sig_loss_and_loh, low_amp, mod_amp, high_amp),
           \(x) mean(x, na.rm = TRUE)),
    .groups = "drop"
  )

drivers        <- runQuery("drivers", sampleIds = patients)
gene_locations <- fread(paste0(REFS_DIR, "gene_locations.txt")) %>%
  select(gene, chromosome, start)

validate_columns(ready, c("chr", "start", "end", "homozygous_del", "neutral_loss_and_loh", "sig_loss_and_loh", "low_amp", "mod_amp", "high_amp"), "circos")

# 1 - Write amp/del circos files

In [ ]:
write_circos_files <- function() {
  ready %>%
    transmute(chr, start, end,
              value = paste(high_amp, mod_amp, low_amp, sep = ",")) %>%
    write_delim(paste0(CIRCOS_DATA, "circos_amps.circos"), delim = " ", col_names = FALSE)

  ready %>%
    transmute(chr, start, end,
              value = paste(-homozygous_del, -sig_loss_and_loh, -neutral_loss_and_loh, sep = ",")) %>%
    write_delim(paste0(CIRCOS_DATA, "circos_dels.circos"), delim = " ", col_names = FALSE)
}

write_circos_files()

# 2 - Gene annotation circos files

In [ ]:
cohort_map <- cohorts %>% select(sampleId, cohort)

gene_map <-
  gene_locations %>%
  mutate(
    bin   = floor(start / 1e6) * 1e6,
    start = as.character(as.integer(bin)),
    end   = as.character(as.integer(bin + 1e6)),
    chr   = paste0("hs", chromosome)
  ) %>%
  select(gene, chr, start, end)

gene_ready <-
  drivers %>%
  filter(driver_simple %in% c("AMPLIFICATION", "DELETION")) %>%
  transmute(sampleId, gene, status = driver_simple) %>%
  inner_join(cohort_map, by = "sampleId") %>%
  group_by(cohort, status, gene) %>%
  summarise(ct = n(), .groups = "drop") %>%
  group_by(cohort, status) %>%
  mutate(rk = row_number(desc(ct))) %>%
  ungroup() %>%
  left_join(gene_map, by = "gene", relationship = "many-to-many") %>%
  drop_na(chr) %>%
  group_by(cohort, status, chr) %>%
  mutate(rk_chr = row_number(desc(ct))) %>%
  filter(rk < 10, rk_chr < 3, ct > 1) %>%
  ungroup() %>%
  mutate(color = ifelse(status == "AMPLIFICATION", "color=dred", "color=vdblue")) %>%
  select(chr, start, end, gene, color)

gene_ready %>%
  distinct(chr, start, end, gene, color) %>%
  fwrite(paste0(CIRCOS_DATA, "circos_amp_del_genes.circos"), sep = " ", col.names = FALSE)

# 3 - Run

In [ ]:
circos_args <- c("-nosvg", "-conf", "circos_conf/go.conf", "-outputdir", READY_FIG, "-outputfile", "circos.png")
system2(command = CIRCOS_BIN, args = circos_args)